<a href="https://colab.research.google.com/github/zackdihel/DA-project-1/blob/main/notebooks/02_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [541]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [542]:
df = pd.read_csv("datatable.csv")
df

,Store ID,Chain,Co-owned,State,South NJ,Central NJ,North NJ,Phil. suburb,Easton,NJ Shore,...,Raise increase ($/hr)_2,Special Program_2,Free/Reduced meals_2,Opening Hour_2,Operating hours_2,P_Soda_2,P_Fry_2,P_Entree_2,Num. Registers_2,Num. Registers at 11am_2
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1,2,6.50,16.50,1.03,.,0.94,4,4
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0,2,10.00,13.00,1.01,0.89,2.35,4,4
2,506,2,1,0,0,0,0,1,0,0,...,0.25,.,1,11.00,11.00,0.95,0.74,2.33,4,3
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,0.92,0.79,0.87,2,2
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,1.01,0.84,0.95,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,423,2,1,1,0,0,1,0,0,0,...,0.50,0,1,11.00,11.00,1.05,0.84,2.32,3,2
406,424,2,1,1,0,0,1,0,0,0,...,0.50,0,1,11.00,14.00,1.05,0.94,2.32,5,3
407,426,3,1,1,0,0,1,0,0,0,...,0.25,1,2,6.00,18.00,1.11,1.05,1.05,6,5
408,427,4,0,1,0,0,1,0,0,0,...,.,1,2,10.50,12.50,1.11,1.09,2.07,2,2


In [543]:
df_filter = df.drop(columns=['Num. Registers','Num. Registers_2','Num. Registers at 11am','Num. Registers at 11am_2',
                 'Opening Hour','Opening Hour_2', 'Operating hours','Operating hours_2',
                 'Num. Callbacks_2','Num. Callbacks','Free/Reduced meals','Free/Reduced meals_2',
                  'Date 2nd Interview','Type 2nd Interview','Special Program_2',
                 'South NJ','Central NJ','Phil. suburb','Easton','NJ Shore','North NJ'])

'State' = 0 PA, 1 NJ;
'Status 2nd Interview' = 0 refused, 1 accepted, 2-5 closed

In [544]:
df_filter = df_filter.replace('.', np.nan)

In [545]:
cols_to_convert = ['FT Empl.', 'PT Empl.', 'FT Empl._2', 'PT Empl._2',
                   'Num. Mgrs', 'Num. Mgrs_2',
                   'Starting Wage', 'Starting Wage_2',
                   'P_Soda', 'P_Fry', 'P_Entree',
                   'P_Soda_2', 'P_Fry_2', 'P_Entree_2',
                   'Status 2nd Interview']
for col in cols_to_convert:
    if col in df_filter.columns:
        df_filter[col] = pd.to_numeric(df_filter[col], errors='coerce')

df_filter['FTE'] = df_filter['FT Empl.'] + df_filter['Num. Mgrs'] + 0.5 * df_filter['PT Empl.']
df_filter['FTE_2'] = df_filter['FT Empl._2'] + df_filter['Num. Mgrs_2'] + 0.5 * df_filter['PT Empl._2']

In [546]:
#closed stores
df_filter['Status 2nd Interview'] = pd.to_numeric(df_filter['Status 2nd Interview'], errors='coerce')
df_filter.loc[df_filter['Status 2nd Interview'].isin([2, 3, 4, 5]), 'FTE_2'] = 0

In [547]:
df_filter['state_label'] = df_filter['State'].map({0: 'PA', 1: 'NJ'})
df_filter = df_filter.rename(columns={
    'FT Empl.' : 'FT_Empl',
    'FT Empl._2' :'FT_Empl_2',
    'Starting Wage' : 'Wage',
    'Starting Wage_2' : 'Wage_2'
})
df_filter

,Store ID,Chain,Co-owned,State,FT_Empl,PT Empl.,Num. Mgrs,Wage,Months to Raise,Raise increase ($/hr),...,Num. Mgrs_2,Wage_2,Months to Raise_2,Raise increase ($/hr)_2,P_Soda_2,P_Fry_2,P_Entree_2,FTE,FTE_2,state_label
0,46,1,0,0,30.0,15.0,3.0,NaN,19.0,NaN,...,3.0,4.30,26.0,0.08,1.03,NaN,0.94,40.50,24.00,PA
1,49,2,0,0,6.5,6.5,4.0,NaN,26.0,NaN,...,4.0,4.45,13.0,0.05,1.01,0.89,2.35,13.75,11.50,PA
2,506,2,1,0,3.0,7.0,2.0,NaN,13.0,0.37,...,4.0,5.00,19.0,0.25,0.95,0.74,2.33,8.50,10.50,PA
3,56,4,1,0,20.0,20.0,4.0,5.00,26.0,0.10,...,2.0,5.25,26.0,0.15,0.92,0.79,0.87,34.00,20.00,PA
4,61,4,1,0,6.0,26.0,5.0,5.50,52.0,0.15,...,6.0,4.75,13.0,0.15,1.01,0.84,0.95,24.00,35.50,PA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,423,2,1,1,2.0,10.0,2.0,4.95,13.0,0.50,...,3.0,5.25,13.0,0.50,1.05,0.84,2.32,9.00,23.75,NJ
406,424,2,1,1,3.5,6.5,3.0,4.75,13.0,0.25,...,4.0,5.25,19.0,0.50,1.05,0.94,2.32,9.75,17.50,NJ
407,426,3,1,1,3.0,33.0,5.0,4.25,26.0,0.15,...,3.0,5.05,19.0,0.25,1.11,1.05,1.05,24.50,20.50,NJ
408,427,4,0,1,7.0,8.0,3.0,4.75,13.0,0.10,...,3.0,5.05,NaN,NaN,1.11,1.09,2.07,14.00,20.50,NJ


In [548]:
# Construct full meal price (soda + fries + entree)
df_filter['FullMeal'] = df_filter['P_Soda'] + df_filter['P_Fry'] + df_filter['P_Entree']
df_filter['FullMeal_2'] = df_filter['P_Soda_2'] + df_filter['P_Fry_2'] + df_filter['P_Entree_2']

In [549]:
#mean and std before NJ min wage increase
before = df_filter.groupby('state_label')[['FTE', 'Wage', 'FullMeal']].agg(['mean','std'])
print("Before wage change:\n\n", before)

Before wage change:

                    FTE                 Wage            FullMeal          
                  mean        std      mean       std      mean       std
state_label                                                              
NJ           20.439408   9.106239  4.612134  0.346351  3.351061  0.644463
PA           23.331169  11.856283  4.630132  0.351687  3.042368  0.601090


In [550]:
#mean and std after NJ min wage increase
after = df_filter.groupby('state_label')[['FTE_2', 'Wage_2', 'FullMeal_2']].agg(['mean','std'])
print("After wage change:\n\n", after)

After wage change:

                  FTE_2              Wage_2           FullMeal_2          
                  mean       std      mean       std       mean       std
state_label                                                              
NJ           20.767028  9.524288  5.080849  0.104545   3.414754  0.635643
PA           21.165584  8.276732  4.617465  0.357478   3.026620  0.566291


In [551]:
after.columns = before.columns

#Difference after wage change
diff = after - before
print("Differences after NJ min. wage change:\n\n", diff)

Differences after NJ min. wage change:

                   FTE                Wage            FullMeal          
                 mean       std      mean       std      mean       std
state_label                                                            
NJ           0.327620  0.418048  0.468715 -0.241806  0.063693 -0.008820
PA          -2.165584 -3.579551 -0.012667  0.005791 -0.015749 -0.034799


In [552]:
#difference between states
state_diff = diff.loc['NJ'] - diff.loc['PA']
print("\nDifference-in-Differences:\n")
print(state_diff)


Difference-in-Differences:

FTE       mean    2.493204
          std     3.997600
Wage      mean    0.481382
          std    -0.247597
FullMeal  mean    0.079442
          std     0.025979
dtype: float64


In [553]:
#difference in means between NJ and PA following NJ's minimum wage increase
diff_means = diff.xs('mean', axis=1, level=1)
print("Difference in means following NJ min. wage increase: \n\n",diff_means)

Difference in means following NJ min. wage increase: 

                   FTE      Wage  FullMeal
state_label                              
NJ           0.327620  0.468715  0.063693
PA          -2.165584 -0.012667 -0.015749


In [554]:
#difference in std between NJ and PA following NJ's minimum wage increase
diff_std = diff.xs('std', axis=1, level=1)
print("Difference in std following NJ min. wage increase: \n\n",diff_std)

Difference in std following NJ min. wage increase: 

                   FTE      Wage  FullMeal
state_label                              
NJ           0.418048 -0.241806 -0.008820
PA          -3.579551  0.005791 -0.034799


In [555]:
import statsmodels.formula.api as smf

df_filter['Treat'] = df_filter['State']

before_df = df_filter[['Store ID', 'Treat', 'FTE', 'Wage', 'FullMeal']].copy()
before_df['Post'] = 0

after_df = df_filter[['Store ID', 'Treat', 'FTE_2', 'Wage_2', 'FullMeal_2']].copy()
after_df['Post'] = 1
after_df = after_df.rename(columns={'FTE_2': 'FTE', 'Wage_2': 'Wage','FullMeal_2': 'FullMeal'})

In [556]:
valid_stores = df_filter.dropna(subset=['FTE', 'FTE_2', 'Wage', 'Wage_2'])['Store ID']

df_combine = pd.concat([before_df, after_df],ignore_index=True)
df_combine_clean = df_combine[df_combine['Store ID'].isin(valid_stores)].dropna().reset_index(drop=True)

In [557]:
print(f"\nStores in regression sample: {df_combine_clean['Store ID'].nunique()}")


Stores in regression sample: 349


In [564]:
df_combine_clean['LogFullMeal'] = np.log(df_combine_clean['FullMeal'])

#OLS regression

model_FTE = smf.ols("FTE ~ Treat * Post", data=df_combine_clean).fit(
    cov_type='cluster',
    cov_kwds={'groups': df_combine_clean['Store ID']}
)
print(model_FTE.summary())

                            OLS Regression Results                            
Dep. Variable:                    FTE   R-squared:                       0.013
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     1.870
Date:                Tue, 10 Mar 2026   Prob (F-statistic):              0.134
Time:                        17:56:48   Log-Likelihood:                -2399.3
No. Observations:                 667   AIC:                             4807.
Df Residuals:                     663   BIC:                             4825.
Df Model:                           3                                         
Covariance Type:              cluster                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.6270      1.539     15.355      0.0

In [562]:
for outcome in ['FTE', 'Wage', 'LogFullMeal']:
    #clusters std err
    model = smf.ols(f'{outcome} ~ Treat * Post', data=df_combine_clean).fit(
        cov_type='cluster',
        cov_kwds={'groups': df_combine_clean['Store ID']}
    )
    print(f"\n--- Regression: {outcome} ---")
    print(model.summary().tables[1])


--- Regression: FTE ---
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.6270      1.539     15.355      0.000      20.611      26.643
Treat         -3.4623      1.613     -2.146      0.032      -6.624      -0.301
Post          -1.8614      1.433     -1.299      0.194      -4.670       0.947
Treat:Post     2.5306      1.528      1.657      0.098      -0.464       5.525

--- Regression: Wage ---
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.6630      0.045    103.065      0.000       4.574       4.752
Treat         -0.0539      0.050     -1.083      0.279      -0.151       0.044
Post          -0.0366      0.050     -0.729      0.466      -0.135       0.062
Treat:Post     0.5083      0.054      9.416      0.000       0.403       0.614

